In [1]:
import random
from pathlib import Path

import numpy as np
import optuna
from seabirdscientific.processing import MinVelocityType, loop_edit_pressure

from ctdam.conv import decode_hex
from ctdam.conv.cast_borders import smoothing
from ctdam.parser import CnvFile
from ctdam.proc.modules.seabird_functions import LoopRemoval

/home/lilith/PycharmProjects/ctdam/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
WEIGHT = 0.67

# load data

In [3]:
data_dir = Path("../../../lr_dataset/").resolve()
out_dir = data_dir.parent / "cnv_out"
out_dir.mkdir(exist_ok=True)

for hex_path in data_dir.glob("**/*.hex"):
    out_path = out_dir / f"{hex_path.stem}.cnv"
    if out_path.exists():
        continue

    try:
        ctd = decode_hex(hex_path)
    except Exception:
        continue

    ctd.to_cnv(out_path)

# process dataset

In [4]:
cnv_dir = Path("../../../cnv_out").resolve()


def load_datasets() -> list[dict]:
    datasets = []

    for path in sorted(cnv_dir.glob("*.cnv")):
        try:
            ctd = CnvFile(path).to_ctd_data()
            pressure = ctd["prDM"].data
            sample_interval = 1.0 / ctd.sample_rate
            datasets.append(
                {
                    "name": path.name,
                    "pressure": pressure,
                    "sample_interval": sample_interval,
                    "latitude": ctd["latitude"].data,
                }
            )
        except Exception as e:
            print(f"skipping {path.name}:{e}")

    return datasets


dataset = load_datasets()

In [5]:
# smooth pressure
for x in dataset:
    x["pressure"] = smoothing(x["pressure"])

In [6]:
def split_dataset(dataset, test_ratio, seed=0):
    rng = random.Random(seed)
    idx = list(range(len(dataset)))
    rng.shuffle(idx)
    n_test = int(round(len(dataset) * test_ratio))
    test_idx = set(idx[:n_test])
    train = [dataset[i] for i in idx if i not in test_idx]
    test = [dataset[i] for i in idx if i in test_idx]
    return train, test

In [7]:
train_set, test_set = split_dataset(dataset, test_ratio=0.3)

# Hyperparameter Optimisiation

In [8]:
def monotonicity_score(
    pressure: np.ndarray, flags: np.ndarray
) -> tuple[float, int]:
    remaining_mask = ~flags
    remaining = pressure[remaining_mask]
    n_flags = int(np.sum(flags))

    if len(remaining) < 2:
        return float("nan"), n_flags

    diffs = np.diff(remaining)
    score = float((diffs > 0).sum()) / len(diffs)
    return score, n_flags

### Jens

In [9]:
import numpy as np
from sklearn.model_selection import KFold


def make_objective_cv_jens(
    dataset, remover, min_len=10, weight=0.5, k=5, seed=0
):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    def obj(trial):
        params = dict(
            precut_period=trial.suggest_int("precut_period", 2, 20),
            cut_period=trial.suggest_int("cut_period", 2, 60),
            mean_speed_percent=trial.suggest_int("mean_speed_percent", 1, 60),
            delay=trial.suggest_int("delay", 1, 10),  # seconds
            filter_order=trial.suggest_categorical(
                "filter_order", [2, 3, 4, 5, 6]
            ),
        )

        fold_scores = []

        for _, val_idx in kf.split(dataset):
            val_fold = [dataset[i] for i in val_idx]

            mono_scores = []
            flag_props = []

            # math stops mathing if i dont include dis
            for item in val_fold:
                pressure = item["pressure"]
                if len(pressure) < min_len:
                    continue

                flags = remover.jens_loop_removal(
                    pressure=pressure,
                    sample_interval=item["sample_interval"],
                    **params,
                )

                score, _ = monotonicity_score(pressure=pressure, flags=flags)
                if np.isnan(score):
                    return -1e9

                mono_scores.append(score)
                flag_props.append(float(np.mean(flags)))

            if not mono_scores:
                return -1e9

            mean_mono = float(np.mean(mono_scores))
            mean_flags_good = 1.0 - float(np.mean(flag_props))
            fold_scores.append(
                weight * mean_mono + (1.0 - weight) * mean_flags_good
            )

        return float(np.mean(fold_scores))

    return obj

In [10]:
def evaluate_jens(dataset, remover, params):
    scores = []
    for item in dataset:
        pressure = item["pressure"]
        sample_interval = item["sample_interval"]
        try:
            flags = remover.jens_loop_removal(
                pressure=pressure, sample_interval=sample_interval, **params
            )
        except ValueError:
            continue
        score, _ = monotonicity_score(pressure=pressure, flags=flags)
        if np.isnan(score):
            continue
        scores.append(score)
    return float(np.mean(scores))

In [11]:
remover = LoopRemoval()

study = optuna.create_study(direction="maximize")  # new study
study.optimize(
    make_objective_cv_jens(
        train_set, remover, min_len=10, weight=WEIGHT, k=5, seed=0
    ),
    n_trials=300,
)

[I 2026-07-09 13:36:52,534] A new study created in memory with name: no-name-b3ae2a75-833a-4ea6-8622-317282151899
/home/lilith/PycharmProjects/ctdam/src/ctdam/proc/modules/seabird_functions.py:144: UserWarning: LoopRemoval is still in an experimental state. Be cautious with the results.
  warnings.warn(
[I 2026-07-09 13:36:52,805] Trial 0 finished with value: 0.9560952873310402 and parameters: {'precut_period': 2, 'cut_period': 15, 'mean_speed_percent': 35, 'delay': 5, 'filter_order': 4}. Best is trial 0 with value: 0.9560952873310402.
[I 2026-07-09 13:36:53,078] Trial 1 finished with value: 0.9557609647230597 and parameters: {'precut_period': 6, 'cut_period': 38, 'mean_speed_percent': 9, 'delay': 6, 'filter_order': 5}. Best is trial 0 with value: 0.9560952873310402.
[I 2026-07-09 13:36:53,356] Trial 2 finished with value: 0.9609612366634706 and parameters: {'precut_period': 11, 'cut_period': 9, 'mean_speed_percent': 33, 'delay': 10, 'filter_order': 6}. Best is trial 2 with value: 0.96

In [12]:
best_params = study.best_params
test_score = evaluate_jens(test_set, remover, best_params)
print("best_params:", best_params)
print("test_score:", test_score)

best_params: {'precut_period': 3, 'cut_period': 59, 'mean_speed_percent': 8, 'delay': 1, 'filter_order': 6}
test_score: 0.9840128630189539


### Seabird

In [13]:
def sample_ef_params(trial):
    return dict(
        window_size=trial.suggest_float("window_size", 0, 60.0),
        mean_speed_percent=trial.suggest_float(
            "mean_speed_percent", 1.0, 100.0
        ),
        min_velocity=trial.suggest_float("min_velocity", 0.0, 1),
        min_soak_depth=trial.suggest_float("min_soak_depth", 0.0, 10.0),
        max_soak_depth=trial.suggest_float("max_soak_depth", 0.0, 20.0),
        remove_surface_soak=trial.suggest_categorical(
            "remove_surface_soak", [True, False]
        ),
        use_deck_pressure_offset=trial.suggest_categorical(
            "use_deck_pressure_offset", [True, False]
        ),
        exclude_flags=trial.suggest_categorical(
            "exclude_flags", [True, False]
        ),
        min_velocity_type=trial.suggest_categorical(
            "min_velocity_type",
            [MinVelocityType.FIXED, MinVelocityType.PERCENT],
        ),
    )


def monotonicity_objective_for_dataset(
    dataset,
    ef_params,
):
    mono_scores = []
    flag_props = []

    for item in dataset:
        pressure = item["pressure"]
        sample_interval = item["sample_interval"]
        latitude = item["latitude"]
        flag = np.array(
            [0.0 for _ in range(len(pressure))], dtype=float
        )  # filler
        try:
            edited_pressure = loop_edit_pressure(
                pressure=pressure,
                latitude=latitude,
                flag=flag,
                sample_interval=sample_interval,
                min_velocity_type=ef_params["min_velocity_type"],
                min_velocity=ef_params["min_velocity"],
                window_size=ef_params["window_size"],
                mean_speed_percent=ef_params["mean_speed_percent"],
                remove_surface_soak=ef_params["remove_surface_soak"],
                min_soak_depth=ef_params["min_soak_depth"],
                max_soak_depth=ef_params["max_soak_depth"],
                use_deck_pressure_offset=ef_params["use_deck_pressure_offset"],
                exclude_flags=ef_params["exclude_flags"],
                flag_value=-9.99e-29,
            )
        except ValueError:
            continue

        score, _ = monotonicity_score(pressure=pressure, flags=edited_pressure)
        if np.isnan(score):
            return None

        mono_scores.append(score)
        flag_props.append(float(np.mean(edited_pressure)))

    if not mono_scores:
        return None

    mean_mono = float(np.mean(mono_scores))
    mean_flags_good = 1.0 - float(np.mean(flag_props))
    return mean_mono, mean_flags_good


def make_objective_cv_sb(dataset, k=5, seed=42, min_len=10, weight=0.5):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    def obj(trial):
        ef_params = sample_ef_params(trial)

        if ef_params["max_soak_depth"] < ef_params["min_soak_depth"]:
            return -1e9

        fold_scores = []
        for _, val_idx in kf.split(dataset):
            val_fold = [dataset[i] for i in val_idx]
            # same as above
            val_fold = [x for x in val_fold if len(x["pressure"]) >= min_len]
            if not val_fold:
                return -1e9

            out = monotonicity_objective_for_dataset(val_fold, ef_params)
            if out is None:
                return -1e9

            mean_mono, mean_flags_good = out
            fold_scores.append(
                weight * mean_mono + (1.0 - weight) * mean_flags_good
            )

        return float(np.mean(fold_scores))

    return obj

In [14]:
study = optuna.create_study(direction="maximize")
study.optimize(
    make_objective_cv_sb(train_set, min_len=10, weight=WEIGHT, k=5, seed=0),
    n_trials=1000,
)

[I 2026-07-09 13:38:13,663] A new study created in memory with name: no-name-60a63cc8-bf3f-4c90-80f1-adf7a68c3426
/tmp/ipykernel_33118/4133983828.py:19: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains MinVelocityType.FIXED which is of type MinVelocityType.
  min_velocity_type=trial.suggest_categorical(
/tmp/ipykernel_33118/4133983828.py:19: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains MinVelocityType.PERCENT which is of type MinVelocityType.
  min_velocity_type=trial.suggest_categorical(
[I 2026-07-09 13:38:15,013] Trial 0 finished with value: 0.6325482534310474 and parameters: {'window_size': 37.40403743952055, 'mean_speed_percent': 57.81173917247903, 'min_velocity': 0.39277440196463265, 'min_soak_depth': 6.76044235809419, 'max_soak_depth': 11.03741039066206, 'remove_surface_soak': True, 'use_deck_pre

In [15]:
def evaluate_sb(dataset, params):
    scores = []
    for item in dataset:
        pressure = item["pressure"]
        sample_interval = item["sample_interval"]
        latitude = item["latitude"]
        flag = np.array([0.0 for _ in range(len(pressure))], dtype=float)
        try:
            flags = loop_edit_pressure(
                pressure=pressure,
                sample_interval=sample_interval,
                latitude=latitude,
                flag=flag,
                **params,
            )
        except ValueError:
            continue
        score, _ = monotonicity_score(pressure=pressure, flags=flags)
        if np.isnan(score):
            continue
        scores.append(score)
    return float(np.mean(scores))

In [16]:
best_params = study.best_params
test_score = evaluate_sb(test_set, best_params)
print("best_params:", best_params)
print("test_score:", test_score)

best_params: {'window_size': 0.09483674512779554, 'mean_speed_percent': 59.21886791858024, 'min_velocity': 0.28974718890408363, 'min_soak_depth': 0.43349221158744844, 'max_soak_depth': 9.711679571711592, 'remove_surface_soak': False, 'use_deck_pressure_offset': True, 'exclude_flags': False, 'min_velocity_type': <MinVelocityType.PERCENT: 1>}
test_score: 0.0003773480686414876


## Time dependent

In [17]:
def make_objective_cv_td(
    dataset, remover, min_len=10, weight=0.5, k=5, seed=0
):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    def obj(trial):
        params = dict(
            delta=trial.suggest_float("delta", 0, 1),
        )

        fold_scores = []

        for _, val_idx in kf.split(dataset):
            val_fold = [dataset[i] for i in val_idx]

            mono_scores = []
            flag_props = []

            # math stops mathing if i dont include dis
            for item in val_fold:
                pressure = item["pressure"]
                if len(pressure) < min_len:
                    continue

                flags = remover.time_dependent_loop_removal(
                    pressure=pressure,
                    delta=params["delta"],
                )

                score, _ = monotonicity_score(pressure=pressure, flags=flags)
                if np.isnan(score):
                    return -1e9

                mono_scores.append(score)
                flag_props.append(float(np.mean(flags)))

            if not mono_scores:
                return -1e9

            mean_mono = float(np.mean(mono_scores))
            mean_flags_good = 1.0 - float(np.mean(flag_props))
            fold_scores.append(
                weight * mean_mono + (1.0 - weight) * mean_flags_good
            )

        return float(np.mean(fold_scores))

    return obj

In [18]:
def evaluate_td(dataset, remover, params):
    scores = []
    for item in dataset:
        pressure = item["pressure"]
        flags = remover.time_dependent_loop_removal(
            pressure=pressure, **params
        )

        score, _ = monotonicity_score(pressure=pressure, flags=flags)
        if np.isnan(score):
            continue
        scores.append(score)
    return float(np.mean(scores))

In [19]:
remover = LoopRemoval()

study = optuna.create_study(direction="maximize")  # new study
study.optimize(
    make_objective_cv_td(
        train_set, remover, min_len=10, weight=WEIGHT, k=5, seed=0
    ),
    n_trials=300,
)

[I 2026-07-09 13:50:55,548] A new study created in memory with name: no-name-86fb9756-06ac-4d76-83bf-8671d1ba056d
[I 2026-07-09 13:50:56,544] Trial 0 finished with value: 0.9630543766991739 and parameters: {'delta': 0.9444982717756023}. Best is trial 0 with value: 0.9630543766991739.
[I 2026-07-09 13:50:57,536] Trial 1 finished with value: 0.9615503019962877 and parameters: {'delta': 0.05849799482781748}. Best is trial 0 with value: 0.9630543766991739.
[I 2026-07-09 13:50:58,502] Trial 2 finished with value: 0.9607950154323474 and parameters: {'delta': 0.046262633271636044}. Best is trial 0 with value: 0.9630543766991739.
[I 2026-07-09 13:50:59,482] Trial 3 finished with value: 0.9525964728848695 and parameters: {'delta': 0.01956388780803453}. Best is trial 0 with value: 0.9630543766991739.
[I 2026-07-09 13:51:00,456] Trial 4 finished with value: 0.9630543766991739 and parameters: {'delta': 0.18384546573100913}. Best is trial 0 with value: 0.9630543766991739.
[I 2026-07-09 13:51:01,446

In [20]:
best_params = study.best_params
test_score = evaluate_td(test_set, remover, best_params)
print("best_params:", best_params)
print("test_score:", test_score)

best_params: {'delta': 0.9444982717756023}
test_score: 0.9481913577363383
